# 🔬 Notebook 04 — Statistical Analysis & Predictive Modelling
### HR Analytics: Job Change of Data Scientists
**Objective:** Hypothesis testing (Chi-square, Mann-Whitney, Point-Biserial), Logistic Regression, Random Forest with feature importances.
**Input:** `data/processed/hr_cleaned.csv` | `hr_engineered.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score, precision_recall_curve)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'primary': '#1B3A6B', 'accent': '#E84855', 'safe': '#2ECC71', 'warn': '#F39C12'}

df     = pd.read_csv('../data/processed/hr_cleaned.csv')
df_eng = pd.read_csv('../data/processed/hr_engineered.csv')
print(f'Analytical df: {df.shape} | Engineered df: {df_eng.shape}')

## 1. Hypothesis Testing Framework

In [ ]:
print('HYPOTHESIS TESTS (α = 0.05)')
print('='*80)
results = []

# ── H1: Chi-Square — City Tier vs Target ─────────────────────────────────────
ct1 = pd.crosstab(df['city_tier'], df['target'])
chi2, p1, dof, _ = chi2_contingency(ct1)
results.append(('H1: City Tier ↔ Job Change',     'Chi-Square', f'χ²={chi2:.2f}', f'p={p1:.4e}', '✅ REJECT' if p1 < 0.05 else '❌ FAIL TO REJECT'))

# ── H2: Chi-Square — Experience Band vs Target ───────────────────────────────
ct2 = pd.crosstab(df['experience_band'], df['target'])
chi2b, p2, _, _ = chi2_contingency(ct2)
results.append(('H2: Experience Band ↔ Job Change','Chi-Square', f'χ²={chi2b:.2f}', f'p={p2:.4e}', '✅ REJECT' if p2 < 0.05 else '❌ FAIL TO REJECT'))

# ── H3: Chi-Square — STEM vs Target ──────────────────────────────────────────
ct3 = pd.crosstab(df['is_stem'], df['target'])
chi2c, p3, _, _ = chi2_contingency(ct3)
results.append(('H3: STEM Major ↔ Job Change',     'Chi-Square', f'χ²={chi2c:.2f}', f'p={p3:.4e}', '✅ REJECT' if p3 < 0.05 else '❌ FAIL TO REJECT'))

# ── H4: Mann-Whitney U — Training Hours: Switchers vs Non-Switchers ──────────
sw  = df[df['target']==1]['training_hours_capped']
nsw = df[df['target']==0]['training_hours_capped']
u_stat, p4 = mannwhitneyu(sw, nsw, alternative='two-sided')
results.append(('H4: Training Hours diff by Target','Mann-Whitney U', f'U={u_stat:.0f}', f'p={p4:.4e}', '✅ REJECT' if p4 < 0.05 else '❌ FAIL TO REJECT'))

# ── H5: Point-Biserial — CDI vs Target ───────────────────────────────────────
rpb, p5 = pointbiserialr(df['target'], df['city_development_index'])
results.append(('H5: CDI correlation with Target',  'Point-Biserial r', f'r={rpb:.4f}', f'p={p5:.4e}', '✅ REJECT' if p5 < 0.05 else '❌ FAIL TO REJECT'))

# ── H6: Chi-Square — Company Size vs Target ───────────────────────────────────
ct6 = pd.crosstab(df['company_size'], df['target'])
chi2d, p6, _, _ = chi2_contingency(ct6)
results.append(('H6: Company Size ↔ Job Change',    'Chi-Square', f'χ²={chi2d:.2f}', f'p={p6:.4e}', '✅ REJECT' if p6 < 0.05 else '❌ FAIL TO REJECT'))

res_df = pd.DataFrame(results, columns=['Hypothesis', 'Test', 'Statistic', 'P-Value', 'Decision'])
print(res_df.to_string(index=False))

## 2. Effect Size Analysis (Cramér's V)

In [ ]:
def cramers_v(x, y):
    """Compute Cramér's V effect size for two categorical variables."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    phi2 = chi2 / n
    r, k = ct.shape
    phi2corr = max(0, phi2 - (k-1)*(r-1)/(n-1))
    rcorr = r - (r-1)**2/(n-1)
    kcorr = k - (k-1)**2/(n-1)
    return np.sqrt(phi2corr / min(kcorr-1, rcorr-1))

cat_features = ['city_tier', 'experience_band', 'education_level', 'company_type',
                'company_size', 'enrolled_university', 'major_discipline', 'gender']

effect_sizes = {}
for col in cat_features:
    effect_sizes[col] = cramers_v(df[col], df['target'])

es_df = pd.Series(effect_sizes).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [COLORS['accent'] if v > 0.1 else COLORS['primary'] for v in es_df.values]
ax.barh(es_df.index, es_df.values, color=colors, alpha=0.88)
ax.axvline(0.1, color=COLORS['warn'], ls='--', lw=1.5, label='Weak effect (0.1)')
ax.axvline(0.3, color=COLORS['accent'], ls='--', lw=1.5, label='Moderate effect (0.3)')
for i, v in enumerate(es_df.values):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel("Cramér's V (Effect Size)")
ax.set_title("Cramér's V — Association Strength with Target (Job Change)", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/04_cramers_v.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Logistic Regression — Interpretable Baseline

In [ ]:
# Prepare data
TARGET = 'target'
X = df_eng.drop(columns=[TARGET])
y = df_eng[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42, C=0.5)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print('LOGISTIC REGRESSION — Classification Report')
print(classification_report(y_test, y_pred_lr, target_names=['Not Switching', 'Switching']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')

# Coefficient plot
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False).head(15)
colors_coef = [COLORS['accent'] if c > 0 else COLORS['primary'] for c in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, alpha=0.88)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Logistic Regression Coefficient')
ax.set_title('Top 15 Feature Coefficients — Logistic Regression\n(Red = increases job-change probability)', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/04_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Random Forest — Feature Importance

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42,
                             max_depth=10, min_samples_leaf=20)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('RANDOM FOREST — Classification Report')
print(classification_report(y_test, y_pred_rf, target_names=['Not Switching', 'Switching']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}')

# Feature importance
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1], color=COLORS['primary'], alpha=0.88)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 15 Feature Importances — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/04_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ROC-AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for name, y_prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0,1], [0,1], color='grey', ls='--', lw=1, label='Random Classifier')
ax.fill_between([0,1], [0,1], alpha=0.05, color='grey')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Model Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('../data/processed/04_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Confusion Matrix & Model Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, y_pred) in zip(axes, [('Logistic Regression', y_pred_lr), ('Random Forest', y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Switching', 'Switching'],
                yticklabels=['Not Switching', 'Switching'])
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('../data/processed/04_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*60)
print('  MODEL COMPARISON SUMMARY')
print('='*60)
print(f'  {'Model':<25} {'F1 (Switch)':<15} {'AUC-ROC'}')
print('-'*60)
print(f'  {'Logistic Regression':<25} {f1_score(y_test, y_pred_lr):<15.4f} {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'  {'Random Forest':<25} {f1_score(y_test, y_pred_rf):<15.4f} {roc_auc_score(y_test, y_prob_rf):.4f}')
print('='*60)
print('\n✅ Statistical analysis complete. Proceeding to 05_final_load_prep.ipynb')

---
## ✅ Next: `05_final_load_prep.ipynb` — Tableau Export Preparation